# 🔍 Hallucination Detection in Small Language Models

This notebook is your workspace for the **SMILES 2026** challenge.  Run the
cells from top to bottom to train and evaluate your hallucination detector.

### What you implement

| File | What to implement |
|------|-------------------|
| `aggregation.py` | `aggregate` — layer selection and token pooling |
| `aggregation.py` | `extract_geometric_features` — optional hand-crafted features |
| `probe.py` | `HallucinationProbe` — probe classifier (`nn.Module` subclass) |
| `splitting.py` | `split_data` — train / validation / test split strategy |

### Fixed infrastructure (do not edit)

| File | Purpose |
|------|--------|
| `model.py` | LLM loader (`get_model_and_tokenizer`) |
| `evaluate.py` | Evaluation loop, summary table, JSON output |


## 0 · Environment Setup

Uncomment and run the block that matches your environment.

In [ ]:
# ── Google Colab ─────────────────────────────────────────────────────────
# Run this block once when opening the notebook in Colab for the first time.

# !git clone https://github.com/ahdr3w/SMILES-HALLUCINATION-DETECTION.git
# %cd SMILES-HALLUCINATION-DETECTION
# !pip install -r requirements.txt

# ── Local (terminal) ─────────────────────────────────────────────────────
# Run these commands in a terminal *before* launching Jupyter, then open
# this notebook from the cloned directory.
#
# git clone https://github.com/ahdr3w/SMILES-HALLUCINATION-DETECTION.git
# cd SMILES-HALLUCINATION-DETECTION
# python -m venv .venv
# source .venv/bin/activate      # Linux / macOS
# # .venv\Scripts\activate.bat   # Windows
# pip install -r requirements.txt
# jupyter notebook solution.ipynb

## 1 · Imports

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from aggregation import aggregation_and_feature_extraction
from evaluate import print_summary, run_evaluation, save_predictions, save_results
from model import MAX_LENGTH, get_model_and_tokenizer
from probe import HallucinationProbe
from splitting import split_data


## 2 · Configuration

In [ ]:
DATA_FILE     = "./data/dataset.csv"   # path to the dataset CSV
OUTPUT_FILE   = "results.json"         # where to write the results summary
BATCH_SIZE    = 4                      # reduce to 1–2 if you run out of GPU memory
USE_GEOMETRIC = False                  # set True to enable geometric feature extraction

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device       : {device}")
print(f"Data         : {DATA_FILE}")
print(f"Max length   : {MAX_LENGTH} tokens")
print(f"Geometric feats: {USE_GEOMETRIC}")


## 3 · Load Dataset

The dataset is a CSV file with three columns:

| Column | Type | Description |
|--------|------|-------------|
| `prompt` | str | Full ChatML-formatted context fed to the language model |
| `response` | str | The model's generated response |
| `label` | float | `1.0` = hallucinated · `0.0` = truthful |

Run the cells below to load the data, inspect the format, and build the
text strings that will be fed to the feature extractor.


In [ ]:
df = pd.read_csv(DATA_FILE)

# Build the text fed to the LLM: concatenation of prompt and response.
all_texts  = [f"{row['prompt']}{row['response']}" for _, row in df.iterrows()]
all_labels = np.array([int(float(h)) for h in df["label"]])

n_total = len(all_labels)
print(f"Loaded {n_total} samples  "
      f"({all_labels.sum()} hallucinated / {(all_labels == 0).sum()} truthful)")

### 3.1 · Data Format — ChatML and Special Tokens

The `prompt` column uses **ChatML** (Chat Markup Language), the conversation
template built into Qwen models.  Each message is wrapped in *role markers*:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
... question and context ...<|im_end|>
<|im_start|>assistant
```

Special tokens and their roles:

- **`<|im_start|>`** — opens a chat turn; the role (`system`, `user`, or `assistant`) immediately follows
- **`<|im_end|>`** — closes the current chat turn
- **`<|endoftext|>`** — end-of-sequence (EOS) token appended by the model at the end of its response

The `prompt` ends right after `<|im_start|>assistant\n` — it provides the
full context *up to* (but not including) the model's reply.  The `response`
column holds the actual generated text, ending with `<|endoftext|>`.

We feed the **concatenation** of `prompt + response` to the feature extractor
so the hidden states capture both the question context *and* the model's
specific answer — the hallucination signal lives in that joint representation.


In [ ]:
# Preview the raw data
print(f"Columns : {df.columns.tolist()}")
print(f"Rows    : {len(df)}")
print(f"Labels  : {dict(df['label'].value_counts().sort_index())}")
print()

# Show the first sample (truncated for readability)
row0 = df.iloc[0]
print("── prompt (first 500 chars) " + "─" * 34)
print(row0["prompt"][:500])
print()
print("── response (first 300 chars) " + "─" * 31)
print(row0["response"][:300])
print()
label_str = "hallucinated" if int(row0["label"]) else "truthful"
print(f"── label : {int(row0['label'])}  ({label_str})")


## 4 · Hidden-State Extraction & Aggregation

The cell below runs the full feature-extraction pipeline in a single
explicit mini-batch loop.  This design has two educational benefits:

1. **Transparency** — you can see every step: tokenisation, the LLM
   forward pass, and how the raw tensor is shaped before aggregation.

2. **Memory efficiency** — `aggregation_and_feature_extraction` is called
   *inside* the loop, immediately after each batch completes.  The large
   raw hidden-state tensors (shape `(n_layers, seq_len, hidden_dim)`) are
   never accumulated; only the compact feature vectors are kept.

The loop structure:

```
for each mini-batch of texts:
    1. tokenise  → input_ids, attention_mask
    2. forward   → outputs.hidden_states  (tuple of n_layers+1 tensors)
    3. stack     → (n_layers, seq_len, hidden_dim) per sample
    4. aggregate → 1-D feature vector per sample   ← your code
    5. append to all_features; raw tensors are freed
```

> ⚠️  This step loads the LLM and runs a forward pass — it may take a few
> minutes depending on your hardware.


In [ ]:
# Load the LLM
model, tokenizer = get_model_and_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.to(device)


In [ ]:
all_features: list = []
t0 = time.time()

for start in tqdm(range(0, len(all_texts), BATCH_SIZE),
                  desc="Extracting & aggregating", unit="batch"):

    # ── 1. Tokenise the current mini-batch ───────────────────────────────
    batch_texts = all_texts[start : start + BATCH_SIZE]
    encoding = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # ── 2. LLM forward pass ──────────────────────────────────────────────
    # outputs.hidden_states: tuple of (n_layers+1) tensors,
    # each with shape (batch, seq_len, hidden_dim).
    # Index 0 → token embeddings; index k → transformer layer k.
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    # ── 3. Stack all layers into one tensor, move to CPU ─────────────────
    # Shape: (batch, n_layers, seq_len, hidden_dim)
    hidden = torch.stack(outputs.hidden_states, dim=1).float()
    mask   = attention_mask.cpu()

    # ── 4. Aggregate each sample and store the compact feature vector ─────
    # The raw `hidden` tensor is released at the end of this loop iteration.
    for i in range(hidden.size(0)):
        feat = aggregation_and_feature_extraction(
            hidden[i],   # (n_layers, seq_len, hidden_dim)
            mask[i],     # (seq_len,)
            use_geometric=USE_GEOMETRIC,
        )
        all_features.append(feat.cpu())

extract_time = time.time() - t0
print(f"Done in {extract_time:.1f} s  —  {len(all_features)} feature vectors extracted")

# Stack into the (N, feature_dim) matrix used by the probe.
X = np.vstack([f.numpy() for f in all_features])   # shape: (N, feature_dim)
y = all_labels                                       # shape: (N,)

print(f"Feature matrix : {X.shape}  (feature_dim = {X.shape[1]})")
print(f"Geometric feats: {USE_GEOMETRIC}")


## 5 · Split Data

The splitting strategy is defined in **`splitting.py`** — edit that file to
change how the dataset is divided (random split, k-fold, group-aware, …).

`split_data` returns a list of `(idx_train, idx_val, idx_test)` tuples.  Pass
`df` as the second argument if your strategy needs per-sample metadata (e.g.
a `question` column for group-aware splits).

In [ ]:
splits = split_data(y, df)

print(f"Splits : {len(splits)} fold(s)")
for i, (tr, va, te) in enumerate(splits):
    print(f"  Fold {i + 1}: train={len(tr)}  "
          f"val={len(va) if va is not None else 'N/A'}  test={len(te)}")

## 6 · Evaluate

For every split the loop trains a majority-class baseline and your
`HallucinationProbe`, then reports Accuracy, F1, and AUROC.

In [ ]:
fold_results = run_evaluation(splits, X, y, HallucinationProbe)

## 7 · Summary & Save Results

In [ ]:
print_summary(fold_results, X.shape[1], len(X), extract_time)
save_results(fold_results, X.shape[1], len(X), extract_time, OUTPUT_FILE)

## 8 · Predict on Test Set

Run the probe on the held-out competition test file (`data/test.csv`) whose
`label` column contains only null values.  The probe is fitted on the
**training and validation data only** — the evaluation test set remains held out
so that the metrics from Section 6 remain an honest estimate of performance.

Predictions are saved to `PREDICTIONS_FILE` (default `predictions.csv`).

In [ ]:
TEST_FILE        = "./data/test.csv"   # competition test set (labels are null)
PREDICTIONS_FILE = "predictions.csv"   # output file with predicted labels

# ── Load test data ────────────────────────────────────────────────────────
df_test    = pd.read_csv(TEST_FILE)
test_texts = [f"{row['prompt']}{row['response']}" for _, row in df_test.iterrows()]
test_ids   = df_test.index
print(f"Test set loaded: {len(test_texts)} samples")

# ── Extract features for test set (same loop as Section 4) ───────────────
test_features: list = []

for start in tqdm(range(0, len(test_texts), BATCH_SIZE),
                  desc="Test extraction & aggregation", unit="batch"):

    batch_texts = test_texts[start : start + BATCH_SIZE]
    encoding = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    hidden = torch.stack(outputs.hidden_states, dim=1).float()
    mask   = attention_mask.cpu()

    for i in range(hidden.size(0)):
        feat = aggregation_and_feature_extraction(
            hidden[i], mask[i], use_geometric=USE_GEOMETRIC,
        )
        test_features.append(feat.cpu())

X_test = np.vstack([f.numpy() for f in test_features])  # (n_test, feature_dim)

# ── Fit final probe on training + validation data only ──────────────────
# Collect the union of all train and validation indices across every split.
# For a single split this excludes idx_test; for k-fold every sample appears
# in a training fold, so all samples are used (same as fitting on X, y).
idx_non_test = np.unique(np.concatenate([
    np.concatenate([idx_tr, idx_va]) if idx_va is not None else idx_tr
    for idx_tr, idx_va, _ in splits
]))
final_probe = HallucinationProbe()
final_probe.fit(X[idx_non_test], y[idx_non_test])

# ── Predict and save ────────────────────────────────────────────────────
save_predictions(final_probe, X_test, test_ids, PREDICTIONS_FILE)
